# Notebook 07 — Function calling + retrieval (Groq)

**Prérequis :** `03` (FAISS).

**Sortie :** `results/function_calling_predictions.json`

**Suite :** `08_ft_plus_rag.ipynb` (FT+RAG), puis **`09_evaluation.ipynb`**


## 0. Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/llm-integration-study/'


## 1. Install


In [ ]:
!pip install -q groq sentence-transformers faiss-cpu


## 2. Run


In [ ]:
import os, json, time, getpass
import numpy as np
import faiss
from groq import Groq
try:
    from groq import BadRequestError as _GroqBadRequest
except Exception:
    _GroqBadRequest = None
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

PROCESSED_PATH = os.path.join(BASE_PATH, 'data', 'processed')
RESULTS_PATH   = os.path.join(BASE_PATH, 'results')
FAISS_PATH     = os.path.join(BASE_PATH, 'models', 'faiss_index')
os.makedirs(RESULTS_PATH, exist_ok=True)

GROQ_MODEL  = "llama-3.1-8b-instant"
EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
TOP_K       = 5
THROTTLE_S  = 0.5

# Schéma minimal (moins de bruit = moins d'échecs tool_use côté Groq)
TOOLS = [{
    "type": "function",
    "function": {
        "name": "search_docs",
        "description": "Semantic search over the indexed corpus. Returns text snippets.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Short search query in French or English."}
            },
            "required": ["query"],
        },
    }
}]

def load_json(p):
    try:
        with open(p, encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print(e)
        return []

test_data   = load_json(os.path.join(PROCESSED_PATH, 'test.json'))
corpus_meta = load_json(os.path.join(FAISS_PATH, 'metadata.json'))
index       = faiss.read_index(os.path.join(FAISS_PATH, 'index.faiss'))
embed_model = SentenceTransformer(EMBED_MODEL)

api_key = getpass.getpass("Clé Groq API : ")
client = Groq(api_key=api_key)

def tool_search_docs(query: str) -> str:
    q = (query or "").strip()[:500] or "."
    q_emb = embed_model.encode([q], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    _, idxs = index.search(q_emb, TOP_K)
    chunks = [corpus_meta[i] for i in idxs[0] if i < len(corpus_meta)]
    parts = []
    for j, c in enumerate(chunks):
        t = (c.get('text', '') or '')[:800]
        parts.append(f"[{j+1} | {c.get('title','')[:50]}] {t}")
    return "\n".join(parts) if parts else "(aucun document)"

def _tool_calls_to_log(tool_calls):
    if not tool_calls:
        return []
    out = []
    for tc in tool_calls:
        out.append({"name": tc.function.name, "arguments": tc.function.arguments})
    return out

def _final_answer_from_docs(question: str, doc_txt: str) -> str:
    """2e appel Groq sans outils (stable) une fois les extraits connus."""
    messages = [
        {"role": "system", "content": "Tu es un assistant. Réponds en français, de façon concise, en t'appuyant uniquement sur les extraits fournis."},
        {"role": "user", "content": f"Question : {question}\n\nExtraits (résultat de search_docs) :\n{doc_txt}\n\nRéponse finale :"},
    ]
    r = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=messages,
        temperature=0.0,
    )
    return (r.choices[0].message.content or "").strip()

def answer_with_tools(question: str):
    """1) tente tool natif Groq (tool_choice=auto). 2) sinon repli: FAISS sur la question puis réponse."""
    t0 = time.time()
    tool_calls_log = []
    doc_txt = None
    used_fallback = False

    sys1 = (
        "When you need evidence, call the function search_docs exactly once with a short JSON argument "
        '{"query": "..."}. Then stop; the user will send tool results in the next turn.'
    )
    messages_round1 = [
        {"role": "system", "content": sys1},
        {"role": "user", "content": question},
    ]

    def try_native(tool_choice):
        return client.chat.completions.create(
            model=GROQ_MODEL,
            messages=messages_round1,
            tools=TOOLS,
            tool_choice=tool_choice,
            temperature=0.0,
        )

    msg = None
    try:
        r1 = try_native("auto")
        msg = r1.choices[0].message
    except Exception as e:
        err = str(e).lower()
        if _GroqBadRequest is not None and isinstance(e, _GroqBadRequest):
            used_fallback = True
            msg = None
        elif "tool_use_failed" in err or "invalid_request_error" in err or " 400 " in err or err.startswith("400"):
            used_fallback = True
            msg = None
        else:
            raise

    if msg is not None and getattr(msg, "tool_calls", None):
        try:
            raw_args = msg.tool_calls[0].function.arguments or "{}"
            args = json.loads(raw_args)
            q = str(args.get("query", question))[:500]
        except json.JSONDecodeError:
            q = question[:500]
            used_fallback = True
        doc_txt = tool_search_docs(q)
        tool_calls_log = _tool_calls_to_log(msg.tool_calls)
        messages_round2 = messages_round1 + [
            {"role": "assistant", "content": msg.content or "", "tool_calls": msg.tool_calls},
            {"role": "tool", "tool_call_id": msg.tool_calls[0].id, "content": doc_txt},
        ]
        # Pas de tools au 2e tour : sinon le modèle retente search_docs et Groq peut lever
        # tool_use_failed (JSON d'arguments mal formé, apostrophes dans la requête, etc.).
        try:
            r2 = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=messages_round2,
                temperature=0.0,
            )
            pred2 = (r2.choices[0].message.content or "").strip()
        except Exception:
            used_fallback = True
            pred2 = (_final_answer_from_docs(question, doc_txt) or "").strip()
        lat = round((time.time() - t0) * 1000)
        return pred2, lat, tool_calls_log, used_fallback

    # Pas d'outil valide ou erreur 400: repli déterministe (même pipeline RAG, sans JSON outil)
    used_fallback = True
    doc_txt = tool_search_docs(question[:500])
    tool_calls_log = [{"name": "search_docs", "arguments": json.dumps({"query": question[:500]}), "fallback": True}]
    pred = (_final_answer_from_docs(question, doc_txt) or "").strip()
    lat = round((time.time() - t0) * 1000)
    return pred, lat, tool_calls_log, used_fallback

fc_predictions = []
fallback_count = 0
for item in tqdm(test_data, desc="Function-calling + RAG"):
    q = item.get('question', '')
    gold = item.get('answer', '')
    pred, lat, tcalls, fb = answer_with_tools(q)
    if fb:
        fallback_count += 1
    fc_predictions.append({
        "pair_id": item.get('pair_id', ''),
        "question": q,
        "predicted_answer": (pred or "").strip(),
        "true_answer": gold,
        "latency_ms": lat,
        "method": "function_calling",
        "dataset_type": item.get("dataset_type", ""),
        "question_type": item.get("question_type", ""),
        "tool_calls": tcalls,
        "tool_native_failed_or_skipped": bool(fb),
    })
    time.sleep(THROTTLE_S)

outp = os.path.join(RESULTS_PATH, 'function_calling_predictions.json')
with open(outp, 'w', encoding='utf-8') as f:
    json.dump(fc_predictions, f, ensure_ascii=False, indent=2)
print("Sauvegardé", outp, len(fc_predictions), "| fallback (sans tool natif valide):", fallback_count)
